In [1]:
import os
import pandas as pd

In [2]:
# -----------------------------
# STEP 0: Project folders (relative paths)
# -----------------------------

# Project root in notebook
project_root = os.getcwd()

raw_folder = os.path.join(project_root, "data", "raw")
processed_folder = os.path.join(project_root, "data", "processed")
db_folder = os.path.join(project_root, "db")

# Create folders if they don't exist
for folder in [raw_folder, processed_folder, db_folder]:
    os.makedirs(folder, exist_ok=True)

print("Folders created/verified!")

Folders created/verified!


In [3]:
# ===============================
# Step 1: Load kaggle Sales Data
# ===============================
# List of filenames
years = [2014, 2015, 2016, 2017]
all_data = []

for year in years:
    file_path = os.path.join(raw_folder, f"orders_{year}.xlsx")
    df = pd.read_excel(file_path)
    df['Year'] = year  # optional, add a column to track the year
    all_data.append(df)

# Combine all years into a single DataFrame
retail_data = pd.concat(all_data, ignore_index=True)

In [4]:
print(retail_data.shape)

(9994, 26)


In [5]:
print(retail_data.head())

   Row ID        Order ID Order Date  Ship Date       Ship Mode Customer ID  \
0       6  CA-2014-115812 2014-06-09 2014-06-14  Standard Class    BH-11710   
1       7  CA-2014-115812 2014-06-09 2014-06-14  Standard Class    BH-11710   
2       8  CA-2014-115812 2014-06-09 2014-06-14  Standard Class    BH-11710   
3       9  CA-2014-115812 2014-06-09 2014-06-14  Standard Class    BH-11710   
4      10  CA-2014-115812 2014-06-09 2014-06-14  Standard Class    BH-11710   

     Customer Name   Segment        Country         City  ...  \
0  Brosina Hoffman  Consumer  United States  Los Angeles  ...   
1  Brosina Hoffman  Consumer  United States  Los Angeles  ...   
2  Brosina Hoffman  Consumer  United States  Los Angeles  ...   
3  Brosina Hoffman  Consumer  United States  Los Angeles  ...   
4  Brosina Hoffman  Consumer  United States  Los Angeles  ...   

                                        Product Name    Sales Quantity  \
0  Eldon Expressions Wood and Plastic Desk Access...   48.86

In [6]:
retail_data.to_csv(os.path.join(raw_folder, "raw_merged_retail_data.csv"), index=False)
print("Raw merged yearly US daily purchases saved:", retail_data.shape)

Raw merged yearly US daily purchases saved: (9994, 26)


In [7]:
print(retail_data.dtypes)

Row ID                          int64
Order ID                       object
Order Date             datetime64[ns]
Ship Date              datetime64[ns]
Ship Mode                      object
Customer ID                    object
Customer Name                  object
Segment                        object
Country                        object
City                           object
State                          object
Postal Code                     int64
Region                         object
Product ID                     object
Category                       object
Sub-Category                   object
Product Name                   object
Sales                         float64
Quantity                        int64
Discount                      float64
Discount value                float64
Profit                        float64
COGS                          float64
Shipping time range             int64
Year sales                      int64
Year                            int64
dtype: objec

In [8]:
#DATA PRE-PROCESSING
#Datatype Handling
retail_data['Order Date'] = pd.to_datetime(retail_data['Order Date'])
# Drop rows with missing Order Date (irelevant for us)
retail_data = retail_data.dropna(subset=['Order Date'])
#Handling na values
retail_data['Quantity'] = retail_data['Quantity'].fillna(0)
retail_data['Sales'] = retail_data['Sales'].fillna(0)

In [9]:
#FETCHING RELEVANT COLUMNS
relevant_retail_columns = ['Order Date', 'Sales', 'Quantity', 'Customer ID',  'Region', 'Product ID', 'Category', 'Discount', 'Discount value', 'Quantity', 'Year']
retail_data = retail_data[relevant_retail_columns]

In [10]:
print(retail_data.columns.tolist())

['Order Date', 'Sales', 'Quantity', 'Customer ID', 'Region', 'Product ID', 'Category', 'Discount', 'Discount value', 'Quantity', 'Year']


In [11]:
#Resolve Duplicate Columns
retail_data = retail_data.loc[:, ~retail_data.columns.duplicated()]

In [12]:
print(retail_data[['Quantity','Sales']].dtypes)

Quantity      int64
Sales       float64
dtype: object


In [13]:
retail_data

,Order Date,Sales,Quantity,Customer ID,Region,Product ID,Category,Discount,Discount value,Year
0,2014-06-09,48.860,7,BH-11710,West,FUR-FU-10001487,Furniture,0.0,0.0000,2014
1,2014-06-09,7.280,4,BH-11710,West,OFF-AR-10002833,Office Supplies,0.0,0.0000,2014
2,2014-06-09,907.152,6,BH-11710,West,TEC-PH-10002275,Technology,0.2,-181.4304,2014
3,2014-06-09,18.504,3,BH-11710,West,OFF-BI-10003910,Office Supplies,0.2,-3.7008,2014
4,2014-06-09,114.900,5,BH-11710,West,OFF-AP-10002892,Office Supplies,0.0,0.0000,2014
...,...,...,...,...,...,...,...,...,...,...
9989,2017-11-17,206.100,5,RA-19885,South,TEC-PH-10004006,Technology,0.0,0.0000,2017
9990,2017-02-26,91.960,2,DB-13060,West,FUR-FU-10000747,Furniture,0.0,0.0000,2017
9991,2017-02-26,258.576,2,DB-13060,West,TEC-PH-10003645,Technology,0.2,-51.7152,2017
9992,2017-02-26,29.600,4,DB-13060,West,OFF-PA-10004041,Office Supplies,0.0,0.0000,2017


In [14]:
daily_orders = retail_data.groupby('Order Date').agg({
    'Quantity': 'sum',
    'Sales': 'sum',
    'Customer ID': 'nunique'  # number of unique customers
}).reset_index()
daily_orders

,Order Date,Quantity,Sales,Customer ID
0,2014-01-03,2,16.4480,1
1,2014-01-04,8,288.0600,1
2,2014-01-05,3,19.5360,1
3,2014-01-06,30,4407.1000,3
4,2014-01-07,10,87.1580,1
...,...,...,...,...
1232,2017-12-26,12,814.5940,4
1233,2017-12-27,6,177.6360,1
1234,2017-12-28,64,1657.3508,10
1235,2017-12-29,41,2915.5340,6


In [15]:
#Save Processed Files Locally
retail_data.to_csv(os.path.join(processed_folder, "cleaned_retail_data.csv"), index=False)
print("Processed US daily purchases saved:", retail_data.shape)

Processed US daily purchases saved: (9994, 10)


In [16]:
retail_data.to_json(os.path.join(processed_folder, "retail_data_clean.json"),
                       orient="records", date_format="iso")
print("Processed USA daily purchases saved:", retail_data.shape)

Processed USA daily purchases saved: (9994, 10)


In [17]:
daily_orders.to_csv(os.path.join(processed_folder, "daily_purchasing_activity.csv"), index=False)
print("Daily purchasing Activity saved:", retail_data.shape)

Daily purchasing Activity saved: (9994, 10)
